<a href="https://colab.research.google.com/github/Leila828/alzheimer_oasis_transfer_learning/blob/main/alzheimer_oasis_V10_%26_v11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ninadaithal_imagesoasis_path = kagglehub.dataset_download('ninadaithal/imagesoasis')
habibleila_vgg16_seed42_weights_path = kagglehub.dataset_download('habibleila/vgg16-seed42-weights')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/habibleila/vgg16-seed42-weights/VGG16_seed42.weights.h5
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_100.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_101.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_102.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_103.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_104.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_105.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_106.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_107.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028_MR1_mpr-1_108.jpg
/kaggle/input/datasets/ninadaithal/imagesoasis/Data/Mild Dementia/OAS1_0028

In [ ]:
# ============================================================================
# COMPLETE KAGGLE NOTEBOOK SCRIPT
# VGG16 CHECKPOINT -> DISK-STREAMED VGG/RESNET FEATURES -> BCAF HEAD
# ============================================================================
# Paste this entire file into ONE Kaggle notebook cell and run it.
#
# Kaggle setup:
#   1. Enable GPU.
#   2. Add the OASIS image dataset as a Kaggle input.
#   3. Add VGG16_seed42.weights.h5 as a Kaggle input.
#
# This script:
#   - discovers the dataset and checkpoint automatically;
#   - creates a subject-level split before sampling slices;
#   - keeps every subject but uses at most MAX_SLICES_PER_SUBJECT images;
#   - evaluates the supplied VGG16+CBAM checkpoint;
#   - extracts VGG and ResNet feature maps to float16 disk memmaps;
#   - trains a BCAF fusion head without keeping all feature maps in RAM;
#   - uses a small validation monitor subset during training;
#   - evaluates the full validation and test sets only at the end;
#   - handles a test split with no Moderate Dementia subjects/images;
#   - reports accuracy, balanced accuracy, macro-F1, ROC-AUC, recall,
#     specificity, confusion matrices, and subject-level metadata.
#
# Scientific rules implemented:
#   - no image from one subject can appear in two splits;
#   - the test set is never used for model selection;
#   - threshold/calibration selection is not performed on the test set;
#   - missing classes in test are reported as unavailable, not fabricated.
# ============================================================================

# =============================
# 1. Imports and configuration
# =============================
import os
import re
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import VGG16, ResNet50

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
    average_precision_score,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# -----------------------------
# Main settings
# -----------------------------
SEED = 42
SPLIT_SEED = 42
IMG_SIZE = 224
IMAGE_BATCH = 32
FEATURE_BATCH = 64
NUM_CLASSES_EXPECTED = 4

# Keep every subject, but cap adjacent slices per subject.
# This is the main memory/compute control.
MAX_SLICES_PER_SUBJECT = 24

# Train only the BCAF/CBAM fusion head over cached feature maps.
BCAF_EPOCHS = 30
BCAF_LR = 1e-3
EARLY_STOPPING_PATIENCE = 6

# Validation predictions inside each epoch are restricted to this many images.
# Full validation is still evaluated at the end.
MONITOR_VAL_IMAGES = 3000

# Automatically detect the class containing both words.
VMD_CLASS_NAME = None

KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/alzheimer_vgg_checkpoint_bcaf")
FEATURE_DIR = OUTPUT_DIR / "feature_memmaps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Leave these as None. Set only if automatic discovery cannot find the inputs.
MANUAL_DATA_ROOT = None
MANUAL_CHECKPOINT = None

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Output directory:", OUTPUT_DIR)

# =============================
# 2. Discover Kaggle inputs
# =============================
def is_image_file(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def find_checkpoint():
    if MANUAL_CHECKPOINT is not None:
        path = Path(MANUAL_CHECKPOINT)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    candidates = list(KAGGLE_INPUT.rglob("VGG16_seed42.weights.h5"))
    if not candidates:
        candidates = list(KAGGLE_INPUT.rglob("*.weights.h5"))
    if not candidates:
        candidates = list(KAGGLE_INPUT.rglob("*.h5"))

    if not candidates:
        raise FileNotFoundError(
            "No .h5 checkpoint was found under /kaggle/input. "
            "Add VGG16_seed42.weights.h5 as a Kaggle input."
        )

    print("Checkpoint candidates:")
    for candidate in candidates:
        print(" ", candidate)
    return candidates[0]


def looks_like_class_root(path):
    try:
        subdirs = [p for p in path.iterdir() if p.is_dir()]
        if len(subdirs) < 2:
            return False
        nonempty = 0
        for folder in subdirs:
            if any(is_image_file(p) for p in folder.iterdir()):
                nonempty += 1
        return nonempty >= 2
    except (PermissionError, OSError):
        return False


def find_data_root():
    if MANUAL_DATA_ROOT is not None:
        path = Path(MANUAL_DATA_ROOT)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    roots = []
    for path in [KAGGLE_INPUT] + [p for p in KAGGLE_INPUT.rglob("*") if p.is_dir()]:
        if looks_like_class_root(path):
            roots.append(path)

    if not roots:
        raise FileNotFoundError(
            "Could not find class folders below /kaggle/input. "
            "Set MANUAL_DATA_ROOT near the top of the script."
        )

    def score(root):
        class_dirs = []
        for folder in root.iterdir():
            if folder.is_dir() and any(is_image_file(p) for p in folder.iterdir()):
                class_dirs.append(folder)
        image_count = sum(sum(1 for p in folder.iterdir() if is_image_file(p)) for folder in class_dirs)
        return (len(class_dirs) == NUM_CLASSES_EXPECTED, image_count)

    roots.sort(key=score, reverse=True)
    return roots[0]


CHECKPOINT_PATH = find_checkpoint()
DATA_ROOT = find_data_root()
print("Selected checkpoint:", CHECKPOINT_PATH)
print("Selected dataset root:", DATA_ROOT)

# =============================
# 3. Dataset table and subject IDs
# =============================
def infer_subject_id(path):
    """Extract an OASIS ID; fallback is used only for nonstandard filenames."""
    stem = Path(path).stem
    patterns = [
        r"(OAS\d+[_-]\d+)",
        r"(sub[-_]?OAS\d+[_-]?\d*)",
        r"(OAS\d+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, stem, flags=re.IGNORECASE)
        if match:
            return match.group(1).lower()

    # Fallback. Inspect the printed subject counts if filenames are unusual.
    return re.split(r"[_\- ]", stem)[0].lower()


classes = sorted(
    [
        folder.name
        for folder in DATA_ROOT.iterdir()
        if folder.is_dir() and any(is_image_file(p) for p in folder.iterdir())
    ]
)

if len(classes) != NUM_CLASSES_EXPECTED:
    raise ValueError(
        f"Expected four class folders, found {len(classes)}: {classes}"
    )

class_to_index = {name: i for i, name in enumerate(classes)}
index_to_class = {i: name for name, i in class_to_index.items()}

print("\nClass order used throughout this experiment:")
for i in range(len(classes)):
    print(f"  {i}: {index_to_class[i]}")

rows = []
for class_name in classes:
    files = sorted(
        p for p in (DATA_ROOT / class_name).rglob("*") if is_image_file(p)
    )
    for path in files:
        rows.append(
            {
                "path": str(path),
                "class_name": class_name,
                "label": class_to_index[class_name],
                "subject": infer_subject_id(path),
            }
        )

df_all = pd.DataFrame(rows)
if df_all.empty:
    raise RuntimeError("No images found.")

print("\nAll-image class counts:")
print(df_all["class_name"].value_counts().sort_index())
print("All images:", len(df_all))
print("All subjects:", df_all["subject"].nunique())

# =============================
# 4. Subject-level split first
# =============================
def subject_split(dataframe, seed=SPLIT_SEED):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    train_idx, temporary_idx = next(
        gss1.split(dataframe, groups=dataframe["subject"])
    )
    train = dataframe.iloc[train_idx].copy()
    temporary = dataframe.iloc[temporary_idx].copy()

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    val_rel, test_rel = next(
        gss2.split(temporary, groups=temporary["subject"])
    )
    val = temporary.iloc[val_rel].copy()
    test = temporary.iloc[test_rel].copy()

    train_subjects = set(train["subject"])
    val_subjects = set(val["subject"])
    test_subjects = set(test["subject"])

    assert train_subjects.isdisjoint(val_subjects)
    assert train_subjects.isdisjoint(test_subjects)
    assert val_subjects.isdisjoint(test_subjects)

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


train_raw, val_raw, test_raw = subject_split(df_all)

print("\nRaw subject-level split:")
for name, part in [("train", train_raw), ("validation", val_raw), ("test", test_raw)]:
    print(
        f"{name:12s}: images={len(part):6d}, subjects={part['subject'].nunique():5d}, "
        f"labels={part['label'].value_counts().sort_index().to_dict()}"
    )

# Check subjects per class explicitly, especially Moderate Dementia.
subject_counts = (
    df_all.groupby(["label", "class_name"])["subject"]
    .nunique()
    .reset_index(name="unique_subjects")
)
print("\nUnique subjects per class in complete dataset:")
print(subject_counts.to_string(index=False))

# =============================
# 5. Cap slices within each split
# =============================
def sample_per_subject(dataframe, max_slices=MAX_SLICES_PER_SUBJECT):
    """Keep all subjects and at most max_slices evenly spaced images per subject."""
    def pick(group):
        group = group.sort_values("path").reset_index(drop=True)
        if len(group) <= max_slices:
            return group
        positions = np.linspace(0, len(group) - 1, max_slices)
        positions = np.round(positions).astype(int)
        return group.iloc[positions]

    return (
        dataframe.groupby("subject", group_keys=False)
        .apply(pick)
        .reset_index(drop=True)
    )


train_df = sample_per_subject(train_raw)
val_df = sample_per_subject(val_raw)
test_df = sample_per_subject(test_raw)

# Save exact metadata used by this experiment.
split_metadata = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)
split_metadata.to_csv(OUTPUT_DIR / "exact_sampled_subject_split.csv", index=False)

print("\nSampled split used for training/evaluation:")
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(
        f"{name:12s}: images={len(part):6d}, subjects={part['subject'].nunique():5d}, "
        f"labels={part['label'].value_counts().sort_index().to_dict()}"
    )
print("Subject leakage check: PASSED")

# =============================
# 6. Image input pipeline
# =============================
def decode_image(path, label, backbone, training=False):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)

    if training:
        # MRI-compatible conservative augmentation.
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=12.0)
        image = tf.image.random_contrast(image, 0.92, 1.08)
        image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + 16, IMG_SIZE + 16)
        image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])

    if backbone == "VGG16":
        image = tf.keras.applications.vgg16.preprocess_input(image)
    elif backbone == "ResNet50":
        image = tf.keras.applications.resnet50.preprocess_input(image)
    else:
        raise ValueError(backbone)

    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return image, tf.cast(label, tf.int32)


def make_image_dataset(part, backbone, training=False, shuffle=False):
    paths = part["path"].values.astype(str)
    labels = part["label"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(part), 2048), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda p, y: decode_image(p, y, backbone, training=training),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(IMAGE_BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

# =============================
# 7. Exact VGG16+CBAM architecture for checkpoint loading
# =============================
@tf.keras.utils.register_keras_serializable(package="AlzheimerCBAM")
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden = max(channels // self.ratio, 1)
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.fc1 = layers.Dense(hidden, activation="relu")
        self.fc2 = layers.Dense(channels)
        self.reshape = layers.Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        a = self.fc2(self.fc1(self.gap(x)))
        b = self.fc2(self.fc1(self.gmp(x)))
        return x * self.reshape(tf.nn.sigmoid(a + b))

    def get_config(self):
        config = super().get_config()
        config.update({"ratio": self.ratio})
        return config


@tf.keras.utils.register_keras_serializable(package="AlzheimerCBAM")
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            1,
            self.kernel_size,
            padding="same",
            activation="sigmoid",
            use_bias=False,
        )
        super().build(input_shape)

    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx = tf.reduce_max(x, axis=-1, keepdims=True)
        return x * self.conv(tf.concat([avg, mx], axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update({"kernel_size": self.kernel_size})
        return config


# =============================
# 8. Load VGG checkpoint and build ResNet extractor
# =============================
def build_vgg_checkpoint_model():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="VGG16_inp")
    base = VGG16(weights=None, include_top=False, input_tensor=inp)
    base.trainable = False
    x = ChannelAttention(ratio=8, name="VGG16_ca")(base.output)
    x = SpatialAttention(kernel_size=7, name="VGG16_sa")(x)
    x = layers.GlobalAveragePooling2D(name="VGG16_gap")(x)
    x = layers.Dense(256, activation="relu", name="VGG16_fc")(x)
    x = layers.Dropout(0.4, name="VGG16_drop")(x)
    out = layers.Dense(NUM_CLASSES_EXPECTED, activation="softmax", name="VGG16_out")(x)
    return models.Model(inp, out, name="VGG16_CBAM")


vgg_checkpoint_model = build_vgg_checkpoint_model()
print("\nVGG checkpoint model parameters:", f"{vgg_checkpoint_model.count_params():,}")

try:
    vgg_checkpoint_model.load_weights(str(CHECKPOINT_PATH))
    print("VGG16 checkpoint loaded successfully.")
except Exception as error:
    print("Checkpoint loading failed. The uploaded .h5 architecture does not match.")
    raise error

# In Keras 3 / TensorFlow 2.20, VGG16 created with input_tensor=inp is
# flattened into the outer model rather than kept as a nested Model layer.
# Therefore, retrieve the last convolutional feature-map layer directly.
# block5_pool has the required shape (7, 7, 512) for 224x224 VGG16 input.
try:
    vgg_feature_layer = vgg_checkpoint_model.get_layer("block5_pool")
except ValueError:
    # Defensive fallback for a differently named but equivalent final layer.
    candidates = [
        layer for layer in vgg_checkpoint_model.layers
        if tuple(layer.output.shape[1:]) == (7, 7, 512)
    ]
    if not candidates:
        raise RuntimeError(
            "Could not locate a VGG16 feature layer with shape (7, 7, 512)."
        )
    vgg_feature_layer = candidates[-1]

# Extract the convolutional feature map, not the classifier output.
vgg_extractor = keras.Model(
    inputs=vgg_checkpoint_model.input,
    outputs=vgg_feature_layer.output,
    name="VGG16_feature_extractor",
)
print("VGG feature shape:", vgg_extractor.output_shape)

# ImageNet ResNet50 is the second independent feature branch.
resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
resnet_base.trainable = False
resnet_extractor = keras.Model(
    inputs=resnet_base.input,
    outputs=resnet_base.output,
    name="ResNet50_feature_extractor",
)
print("ResNet feature shape:", resnet_extractor.output_shape)

# =============================
# 9. Baseline VGG checkpoint evaluation
# =============================
def predict_model(model, dataset):
    return model.predict(dataset, verbose=1)


y_val = val_df["label"].values.astype(np.int32)
y_test = test_df["label"].values.astype(np.int32)

vgg_val_ds = make_image_dataset(val_df, "VGG16", training=False, shuffle=False)
vgg_test_ds = make_image_dataset(test_df, "VGG16", training=False, shuffle=False)

vgg_baseline_val_probs = predict_model(vgg_checkpoint_model, vgg_val_ds)
vgg_baseline_test_probs = predict_model(vgg_checkpoint_model, vgg_test_ds)

# =============================
# 10. Stream feature maps to float16 memmaps
# =============================
def extract_to_memmap(extractor, dataset, n_samples, path):
    output_shape = tuple(int(x) for x in extractor.output_shape[1:])
    print(f"Extracting {n_samples} samples -> {path}")
    print("Feature shape:", output_shape)
    print("Approximate float16 size (GB):", n_samples * np.prod(output_shape) * 2 / 1e9)

    mm = np.lib.format.open_memmap(
        path,
        mode="w+",
        dtype=np.float16,
        shape=(n_samples, *output_shape),
    )
    cursor = 0
    for images, _ in dataset:
        features = extractor(images, training=False).numpy().astype(np.float16)
        end = cursor + len(features)
        mm[cursor:end] = features
        cursor = end
        if cursor % (FEATURE_BATCH * 10) == 0 or cursor == n_samples:
            print(f"  {cursor}/{n_samples}")
    if cursor != n_samples:
        raise RuntimeError(f"Expected {n_samples}, extracted {cursor}")
    mm.flush()
    del mm
    gc.collect()


# Use sampled dataframes. Images are not held in memory after extraction.
train_vgg_ds = make_image_dataset(train_df, "VGG16", training=False, shuffle=False)
val_vgg_ds = make_image_dataset(val_df, "VGG16", training=False, shuffle=False)
test_vgg_ds = make_image_dataset(test_df, "VGG16", training=False, shuffle=False)

train_res_ds = make_image_dataset(train_df, "ResNet50", training=False, shuffle=False)
val_res_ds = make_image_dataset(val_df, "ResNet50", training=False, shuffle=False)
test_res_ds = make_image_dataset(test_df, "ResNet50", training=False, shuffle=False)

train_vgg_path = FEATURE_DIR / "train_vgg.npy"
val_vgg_path = FEATURE_DIR / "val_vgg.npy"
test_vgg_path = FEATURE_DIR / "test_vgg.npy"
train_res_path = FEATURE_DIR / "train_resnet.npy"
val_res_path = FEATURE_DIR / "val_resnet.npy"
test_res_path = FEATURE_DIR / "test_resnet.npy"

extract_to_memmap(vgg_extractor, train_vgg_ds, len(train_df), train_vgg_path)
extract_to_memmap(vgg_extractor, val_vgg_ds, len(val_df), val_vgg_path)
extract_to_memmap(vgg_extractor, test_vgg_ds, len(test_df), test_vgg_path)

# Release the VGG model before the larger ResNet extraction.
del train_vgg_ds, val_vgg_ds, test_vgg_ds, vgg_checkpoint_model, vgg_extractor
keras.backend.clear_session()
gc.collect()

extract_to_memmap(resnet_extractor, train_res_ds, len(train_df), train_res_path)
extract_to_memmap(resnet_extractor, val_res_ds, len(val_df), val_res_path)
extract_to_memmap(resnet_extractor, test_res_ds, len(test_df), test_res_path)

del train_res_ds, val_res_ds, test_res_ds, resnet_base, resnet_extractor
keras.backend.clear_session()
gc.collect()

# =============================
# 11. Memmap-backed feature datasets
# =============================
train_vgg_mm = np.load(train_vgg_path, mmap_mode="r")
val_vgg_mm = np.load(val_vgg_path, mmap_mode="r")
test_vgg_mm = np.load(test_vgg_path, mmap_mode="r")
train_res_mm = np.load(train_res_path, mmap_mode="r")
val_res_mm = np.load(val_res_path, mmap_mode="r")
test_res_mm = np.load(test_res_path, mmap_mode="r")


def make_feature_dataset(vgg_mm, res_mm, labels, indices=None, batch=FEATURE_BATCH, shuffle=False, sample_weights=None):
    labels = np.asarray(labels, dtype=np.int32)
    if indices is None:
        indices = np.arange(len(labels), dtype=np.int64)
    else:
        indices = np.asarray(indices, dtype=np.int64)

    if sample_weights is None:
        sample_weights = np.ones(len(labels), dtype=np.float32)
    else:
        sample_weights = np.asarray(sample_weights, dtype=np.float32)

    n = len(indices)

    def generator():
        order = np.random.permutation(n) if shuffle else np.arange(n)
        for start in range(0, n, batch):
            selected = order[start:start + batch]
            source_indices = indices[selected]
            yield (
                (
                    np.asarray(vgg_mm[source_indices], dtype=np.float32),
                    np.asarray(res_mm[source_indices], dtype=np.float32),
                ),
                labels[selected],
                sample_weights[selected],
            )

    signature = (
        (
            tf.TensorSpec(shape=(None, 7, 7, 512), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 7, 7, 2048), dtype=tf.float32),
        ),
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
    return tf.data.Dataset.from_generator(generator, output_signature=signature).prefetch(2)


# =============================
# 12. Build the BCAF head
# =============================
@tf.keras.utils.register_keras_serializable(package="AlzheimerBCAF")
class BCAF(layers.Layer):
    def __init__(self, dim=128, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads

    def build(self, input_shapes):
        self.proj_v = layers.Dense(self.dim, use_bias=False, name="proj_v")
        self.proj_r = layers.Dense(self.dim, use_bias=False, name="proj_r")
        self.mha_vr = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.dim // self.num_heads,
            dropout=0.10,
            name="mha_vr",
        )
        self.mha_rv = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.dim // self.num_heads,
            dropout=0.10,
            name="mha_rv",
        )
        self.ln_v = layers.LayerNormalization(name="ln_v")
        self.ln_r = layers.LayerNormalization(name="ln_r")
        super().build(input_shapes)

    def call(self, inputs, training=False):
        f_v, f_r = inputs
        batch_size = tf.shape(f_v)[0]
        height = tf.shape(f_v)[1]
        width = tf.shape(f_v)[2]
        tokens = height * width

        s_v = tf.reshape(self.proj_v(f_v), [batch_size, tokens, self.dim])
        s_r = tf.reshape(self.proj_r(f_r), [batch_size, tokens, self.dim])

        attended_v = self.mha_vr(
            query=s_v, key=s_r, value=s_r, training=training
        )
        attended_r = self.mha_rv(
            query=s_r, key=s_v, value=s_v, training=training
        )

        out_v = self.ln_v(s_v + attended_v)
        out_r = self.ln_r(s_r + attended_r)

        return (
            tf.reshape(out_v, [batch_size, height, width, self.dim]),
            tf.reshape(out_r, [batch_size, height, width, self.dim]),
        )

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim, "num_heads": self.num_heads})
        return config


def build_bcaf_head(n_classes=NUM_CLASSES_EXPECTED, dim=128, heads=4):
    in_v = layers.Input((7, 7, 512), name="vgg_features")
    in_r = layers.Input((7, 7, 2048), name="resnet_features")

    f_v = ChannelAttention(8, name="vgg_feature_ca")(in_v)
    f_v = SpatialAttention(7, name="vgg_feature_sa")(f_v)
    f_r = ChannelAttention(8, name="res_feature_ca")(in_r)
    f_r = SpatialAttention(7, name="res_feature_sa")(f_r)

    fused_v, fused_r = BCAF(dim, heads, name="bcaf")([f_v, f_r])

    x_v = layers.GlobalAveragePooling2D(name="vgg_fused_gap")(fused_v)
    x_r = layers.GlobalAveragePooling2D(name="res_fused_gap")(fused_r)
    x = layers.Concatenate(name="fusion_concat")([x_v, x_r])
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name="fusion_fc",
    )(x)
    x = layers.Dropout(0.40, name="fusion_dropout")(x)
    out = layers.Dense(n_classes, activation="softmax", dtype="float32", name="fusion_out")(x)
    return keras.Model([in_v, in_r], out, name="BCAF_head")


bcaf_model = build_bcaf_head()
print("\nBCAF head parameters:", f"{bcaf_model.count_params():,}")

# =============================
# 13. Monitor validation subset
# =============================
def make_validation_monitor_indices(part, max_images=MONITOR_VAL_IMAGES):
    """Select complete subjects, approximately stratified by class."""
    rng = np.random.default_rng(SEED)
    selected_subjects = []
    total = 0

    # First include subjects from each available class.
    for label in sorted(part["label"].unique()):
        subject_list = part.loc[part["label"] == label, "subject"].unique().tolist()
        rng.shuffle(subject_list)
        if subject_list:
            selected_subjects.append(subject_list[0])

    remaining = [s for s in part["subject"].unique().tolist() if s not in selected_subjects]
    rng.shuffle(remaining)
    selected_subjects.extend(remaining)

    chosen = []
    for subject in selected_subjects:
        count = int((part["subject"] == subject).sum())
        if chosen and total + count > max_images:
            break
        chosen.append(subject)
        total += count
        if total >= max_images:
            break

    indices = np.flatnonzero(part["subject"].isin(chosen).values)
    return indices


val_monitor_indices = make_validation_monitor_indices(val_df)
print("Validation monitor images:", len(val_monitor_indices))
print("Validation full images:", len(val_df))

# =============================
# 14. Class weights and BCAF training
# =============================
y_train = train_df["label"].values.astype(np.int32)
y_val = val_df["label"].values.astype(np.int32)
y_test = test_df["label"].values.astype(np.int32)

present_train_classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=present_train_classes,
    y=y_train,
)
class_weight_map = {int(c): float(w) for c, w in zip(present_train_classes, weights)}
train_sample_weights = np.array([class_weight_map[int(y)] for y in y_train], dtype=np.float32)
print("Training class weights:", class_weight_map)

train_feature_ds = make_feature_dataset(
    train_vgg_mm,
    train_res_mm,
    y_train,
    batch=FEATURE_BATCH,
    shuffle=True,
    sample_weights=train_sample_weights,
)
val_monitor_ds = make_feature_dataset(
    val_vgg_mm,
    val_res_mm,
    y_val[val_monitor_indices],
    indices=val_monitor_indices,
    batch=FEATURE_BATCH,
    shuffle=False,
)

bcaf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=BCAF_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

bcaf_checkpoint = OUTPUT_DIR / "BCAF_head_best.weights.h5"
train_callbacks = [
    callbacks.ModelCheckpoint(
        str(bcaf_checkpoint),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    callbacks.CSVLogger(str(OUTPUT_DIR / "bcaf_training_history.csv")),
]

print("\nStarting BCAF head training...")
history = bcaf_model.fit(
    train_feature_ds,
    validation_data=val_monitor_ds,
    epochs=BCAF_EPOCHS,
    callbacks=train_callbacks,
    verbose=1,
)

if bcaf_checkpoint.exists():
    bcaf_model.load_weights(str(bcaf_checkpoint))

bcaf_model.save(OUTPUT_DIR / "BCAF_head_best.keras")
pd.DataFrame(history.history).to_csv(
    OUTPUT_DIR / "bcaf_training_history_full.csv", index=False
)

# =============================
# 15. Full validation/test prediction
# =============================
def predict_feature_dataset(model, vgg_mm, res_mm, labels):
    dataset = make_feature_dataset(
        vgg_mm,
        res_mm,
        labels,
        batch=FEATURE_BATCH,
        shuffle=False,
    )
    return model.predict(dataset, verbose=1)


bcaf_val_probs = predict_feature_dataset(bcaf_model, val_vgg_mm, val_res_mm, y_val)
bcaf_test_probs = predict_feature_dataset(bcaf_model, test_vgg_mm, test_res_mm, y_test)

# =============================
# 16. Metrics
# =============================
def find_vmd_index():
    if VMD_CLASS_NAME is not None:
        return class_to_index.get(VMD_CLASS_NAME)
    matches = [
        name for name in classes
        if "very" in name.lower() and "mild" in name.lower()
    ]
    return class_to_index[matches[0]] if matches else None


VMD_INDEX = find_vmd_index()


def specificity_by_class(y_true, y_pred):
    labels = list(range(NUM_CLASSES_EXPECTED))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    output = {}
    for label, class_name in index_to_class.items():
        tp = cm[label, label]
        fn = cm[label, :].sum() - tp
        fp = cm[:, label].sum() - tp
        tn = cm.sum() - tp - fn - fp
        output[class_name] = float(tn / (tn + fp)) if (tn + fp) else None
    return output


def multiclass_auc(y_true, probs):
    try:
        present = np.unique(y_true)
        if len(present) < 2:
            return None
        return float(
            roc_auc_score(
                y_true,
                probs,
                labels=list(range(NUM_CLASSES_EXPECTED)),
                multi_class="ovr",
                average="macro",
            )
        )
    except Exception:
        return None


def multiclass_average_precision(y_true, probs):
    try:
        present = np.unique(y_true)
        if len(present) < 2:
            return None
        one_hot = tf.keras.utils.to_categorical(y_true, NUM_CLASSES_EXPECTED)
        return float(average_precision_score(one_hot, probs, average="macro"))
    except Exception:
        return None


def evaluate(y_true, probs, split_name, model_name):
    labels = list(range(NUM_CLASSES_EXPECTED))
    preds = np.argmax(probs, axis=1)
    present = np.unique(y_true).tolist()
    present_names = [index_to_class[int(i)] for i in present]

    report_all = classification_report(
        y_true,
        preds,
        labels=labels,
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )
    report_present = classification_report(
        y_true,
        preds,
        labels=present,
        target_names=present_names,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, preds, labels=labels)

    result = {
        "model": model_name,
        "split": split_name,
        "n_images": int(len(y_true)),
        "classes_present": present_names,
        "missing_classes": [index_to_class[i] for i in labels if i not in present],
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy_present_classes": float(
            balanced_accuracy_score(y_true, preds)
        ),
        "macro_f1_present_classes": float(
            f1_score(y_true, preds, labels=present, average="macro", zero_division=0)
        ),
        "macro_f1_all_four_labels": float(
            f1_score(y_true, preds, labels=labels, average="macro", zero_division=0)
        ),
        "log_loss": float(log_loss(y_true, probs, labels=labels)),
        "roc_auc_macro_ovr": multiclass_auc(y_true, probs),
        "average_precision_macro_ovr": multiclass_average_precision(y_true, probs),
        "per_class": {},
        "confusion_matrix_all_four_labels": cm.tolist(),
        "classification_report_present_classes": report_present,
    }

    specificity = specificity_by_class(y_true, preds)
    for label, class_name in index_to_class.items():
        if class_name in report_all:
            result["per_class"][class_name] = {
                "precision": float(report_all[class_name]["precision"]),
                "recall": float(report_all[class_name]["recall"]),
                "f1": float(report_all[class_name]["f1-score"]),
                "specificity": specificity[class_name],
                "status": "evaluated" if label in present else "not_available_in_split",
            }

    if VMD_INDEX is not None and VMD_INDEX in present:
        result["vmd_recall"] = float(report_all[index_to_class[VMD_INDEX]]["recall"])
    else:
        result["vmd_recall"] = None

    path = OUTPUT_DIR / f"{model_name}_{split_name}_metrics.json"
    with open(path, "w") as handle:
        json.dump(result, handle, indent=2)

    print("\n" + "=" * 78)
    print(f"{model_name} — {split_name}")
    print("=" * 78)
    print("Images:", len(y_true))
    print("Classes present:", present_names)
    print("Missing classes:", result["missing_classes"])
    print("Accuracy:", f"{result['accuracy']:.4f}")
    print("Balanced accuracy on present classes:", f"{result['balanced_accuracy_present_classes']:.4f}")
    print("Macro-F1 on present classes:", f"{result['macro_f1_present_classes']:.4f}")
    print("Macro-F1 treating missing classes as zero:", f"{result['macro_f1_all_four_labels']:.4f}")
    print("Macro ROC-AUC:", result["roc_auc_macro_ovr"])
    print("VMD recall:", result["vmd_recall"])
    print("\nClassification report for classes present in this split:")
    print(classification_report(y_true, preds, labels=present, target_names=present_names, zero_division=0))
    print("Confusion matrix, all four labels:")
    print(cm)
    return result


def save_confusion_plot(y_true, probs, title, filename):
    preds = np.argmax(probs, axis=1)
    cm = confusion_matrix(y_true, preds, labels=list(range(NUM_CLASSES_EXPECTED)))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=250)
    plt.close()


# =============================
# 17. Evaluate VGG baseline and BCAF
# =============================
vgg_baseline_val_metrics = evaluate(
    y_val, vgg_baseline_val_probs, "validation", "vgg_checkpoint"
)
vgg_baseline_test_metrics = evaluate(
    y_test, vgg_baseline_test_probs, "test", "vgg_checkpoint"
)
bcaf_val_metrics = evaluate(y_val, bcaf_val_probs, "validation", "bcaf_head")
bcaf_test_metrics = evaluate(y_test, bcaf_test_probs, "test", "bcaf_head")

save_confusion_plot(
    y_test,
    vgg_baseline_test_probs,
    "VGG16+CBAM checkpoint — subject-level test",
    "vgg_checkpoint_test_confusion_matrix.png",
)
save_confusion_plot(
    y_test,
    bcaf_test_probs,
    "BCAF head — subject-level test",
    "bcaf_test_confusion_matrix.png",
)

# Training curves.
history_df = pd.DataFrame(history.history)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df["loss"], label="train")
plt.plot(history_df["val_loss"], label="monitor validation")
plt.title("BCAF training loss")
plt.xlabel("Epoch")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_df["accuracy"], label="train")
plt.plot(history_df["val_accuracy"], label="monitor validation")
plt.title("BCAF training accuracy")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bcaf_training_curves.png", dpi=250)
plt.close()

# =============================
# 18. Save final summary
# =============================
summary = {
    "configuration": {
        "seed": SEED,
        "split_seed": SPLIT_SEED,
        "max_slices_per_subject": MAX_SLICES_PER_SUBJECT,
        "monitor_val_images": MONITOR_VAL_IMAGES,
        "classes": classes,
        "vmd_index": VMD_INDEX,
        "checkpoint": str(CHECKPOINT_PATH),
        "data_root": str(DATA_ROOT),
    },
    "dataset": {
        "all_images": int(len(df_all)),
        "all_subjects": int(df_all["subject"].nunique()),
        "train_images": int(len(train_df)),
        "validation_images": int(len(val_df)),
        "test_images": int(len(test_df)),
        "train_subjects": int(train_df["subject"].nunique()),
        "validation_subjects": int(val_df["subject"].nunique()),
        "test_subjects": int(test_df["subject"].nunique()),
    },
    "vgg_checkpoint_validation": vgg_baseline_val_metrics,
    "vgg_checkpoint_test": vgg_baseline_test_metrics,
    "bcaf_validation": bcaf_val_metrics,
    "bcaf_test": bcaf_test_metrics,
}

with open(OUTPUT_DIR / "final_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

comparison = pd.DataFrame(
    [
        {
            "model": "VGG16 checkpoint",
            "split": "test",
            "accuracy": vgg_baseline_test_metrics["accuracy"],
            "balanced_accuracy_present": vgg_baseline_test_metrics["balanced_accuracy_present_classes"],
            "macro_f1_present": vgg_baseline_test_metrics["macro_f1_present_classes"],
            "macro_roc_auc": vgg_baseline_test_metrics["roc_auc_macro_ovr"],
            "vmd_recall": vgg_baseline_test_metrics["vmd_recall"],
        },
        {
            "model": "BCAF head",
            "split": "test",
            "accuracy": bcaf_test_metrics["accuracy"],
            "balanced_accuracy_present": bcaf_test_metrics["balanced_accuracy_present_classes"],
            "macro_f1_present": bcaf_test_metrics["macro_f1_present_classes"],
            "macro_roc_auc": bcaf_test_metrics["roc_auc_macro_ovr"],
            "vmd_recall": bcaf_test_metrics["vmd_recall"],
        },
    ]
)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
print("\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))

# Save probabilities for future statistical analysis.
np.save(OUTPUT_DIR / "vgg_baseline_val_probs.npy", vgg_baseline_val_probs)
np.save(OUTPUT_DIR / "vgg_baseline_test_probs.npy", vgg_baseline_test_probs)
np.save(OUTPUT_DIR / "bcaf_val_probs.npy", bcaf_val_probs)
np.save(OUTPUT_DIR / "bcaf_test_probs.npy", bcaf_test_probs)

print("\n" + "=" * 78)
print("COMPLETE")
print("=" * 78)
print("All outputs saved to:", OUTPUT_DIR)
print("The test split is final and was not used for training or model selection.")
print("If Moderate Dementia is absent from test, its test metrics are correctly marked unavailable.")
print("Repeat with SPLIT_SEED 123 and 777 before making a publication claim.")


TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Output directory: /kaggle/working/alzheimer_vgg_checkpoint_bcaf
Checkpoint candidates:
  /kaggle/input/datasets/habibleila/vgg16-seed42-weights/VGG16_seed42.weights.h5
Selected checkpoint: /kaggle/input/datasets/habibleila/vgg16-seed42-weights/VGG16_seed42.weights.h5
Selected dataset root: /kaggle/input/datasets/ninadaithal/imagesoasis/Data

Class order used throughout this experiment:
  0: Mild Dementia
  1: Moderate Dementia
  2: Non Demented
  3: Very mild Dementia

All-image class counts:
class_name
Mild Dementia          5002
Moderate Dementia       488
Non Demented          67222
Very mild Dementia    13725
Name: count, dtype: int64
All images: 86437
All subjects: 347

Raw subject-level split:
train       : images= 59414, subjects=  242, labels={0: 3843, 1: 244, 2: 45201, 3: 10126}
validation  : images= 13298, subject

I0000 00:00:1792439764.019549     143 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


39/39 ━━━━━━━━━━━━━━━━━━━━ 48s 694ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step 
Extracting 5808 samples -> /kaggle/working/alzheimer_vgg_checkpoint_bcaf/feature_memmaps/train_vgg.npy
Feature shape: (7, 7, 512)
Approximate float16 size (GB): 0.291422208
  640/5808
  1280/5808
  1920/5808
  2560/5808
  3200/5808
  3840/5808
  4480/5808
  5120/5808
  5760/5808
  5808/5808
Extracting 1248 samples -> /kaggle/working/alzheimer_vgg_checkpoint_bcaf/feature_memmaps/val_vgg.npy
Feature shape: (7, 7, 512)
Approximate float16 size (GB): 0.062619648
  640/1248
  1248/1248
Extracting 1272 samples -> /kaggle/working/alzheimer_vgg_checkpoint_bcaf/feature_memmaps/test_vgg.npy
Feature shape: (7, 7, 512)
Approximate float16 size (GB): 0.063823872
  640/1272
  1272/1272
Extracting 5808 samples -> /kaggle/working/alzheimer_vgg_checkpoint_bcaf/feature_memmaps/train_resnet.npy
Feature shape: (7, 7, 2048)
Approximate float16 size (GB): 1.165688832
  640/5808
  1280/5808
  1920/5808
  2560/5808
  3200/5808
 

**Version 11**
This version includes the corrections discussed:
Reuses the learned VGG16_ca and VGG16_sa attention features from your .h5 checkpoint.
Does not apply a second randomly initialized VGG attention block.
Reduces the BCAF learning rate:
Python
BCAF_LR = 1e-4
Replaces the extreme automatic Moderate Dementia weight of approximately 60.5.
Uses milder class weights:
Python
Mild Dementia:      2.0
Moderate Dementia:  4.0
Non Demented:       0.5
Very mild Dementia: 1.5
Keeps subject-level splitting and slice-per-subject sampling.
Keeps disk-backed feature extraction to avoid the RAM crash.
Trains the BCAF head with early stopping.
Evaluates the VGG checkpoint and BCAF on the same test set.
What to do in Kaggle
Restart the Kaggle session.
Create one empty code cell.
Paste the entire contents of the revised file.
Run it from the beginning.
Do not paste it below the previously crashed run in the same active session. Restarting is important because the previous session may still contain TensorFlow models and GPU memory allocations.
You should now see:
text
VGG attention feature shape: (None, 7, 7, 512)
ResNet feature shape: (None, 7, 7, 2048)
The most important comparison will be:
text
VGG16 checkpoint test macro-F1
BCAF test macro-F1
VGG16 checkpoint test balanced accuracy
BCAF test balanced accuracy
VGG16 checkpoint VMD recall
BCAF VMD recall
The BCAF model should only be considered better if it improves macro-F1 or balanced accuracy without an unacceptable loss in Non Demented specificity. An increase in Very Mild Dementia recall alone is not sufficient.

In [ ]:
# ============================================================================
# COMPLETE KAGGLE NOTEBOOK SCRIPT
# VGG16 CHECKPOINT -> DISK-STREAMED VGG/RESNET FEATURES -> BCAF HEAD
# ============================================================================
# Paste this entire file into ONE Kaggle notebook cell and run it.
#
# Kaggle setup:
#   1. Enable GPU.
#   2. Add the OASIS image dataset as a Kaggle input.
#   3. Add VGG16_seed42.weights.h5 as a Kaggle input.
#
# This script:
#   - discovers the dataset and checkpoint automatically;
#   - creates a subject-level split before sampling slices;
#   - keeps every subject but uses at most MAX_SLICES_PER_SUBJECT images;
#   - evaluates the supplied VGG16+CBAM checkpoint;
#   - extracts VGG and ResNet feature maps to float16 disk memmaps;
#   - trains a BCAF fusion head without keeping all feature maps in RAM;
#   - uses a small validation monitor subset during training;
#   - evaluates the full validation and test sets only at the end;
#   - handles a test split with no Moderate Dementia subjects/images;
#   - reports accuracy, balanced accuracy, macro-F1, ROC-AUC, recall,
#     specificity, confusion matrices, and subject-level metadata.
#
# Scientific rules implemented:
#   - no image from one subject can appear in two splits;
#   - the test set is never used for model selection;
#   - threshold/calibration selection is not performed on the test set;
#   - missing classes in test are reported as unavailable, not fabricated.
# ============================================================================

# =============================
# 1. Imports and configuration
# =============================
import os
import re
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import VGG16, ResNet50

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
    average_precision_score,
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# -----------------------------
# Main settings
# -----------------------------
SEED = 42
SPLIT_SEED = 42
IMG_SIZE = 224
IMAGE_BATCH = 32
FEATURE_BATCH = 64
NUM_CLASSES_EXPECTED = 4

# Keep every subject, but cap adjacent slices per subject.
# This is the main memory/compute control.
MAX_SLICES_PER_SUBJECT = 24

# Train only the BCAF/CBAM fusion head over cached feature maps.
BCAF_EPOCHS = 30
BCAF_LR = 1e-4
EARLY_STOPPING_PATIENCE = 6

# Validation predictions inside each epoch are restricted to this many images.
# Full validation is still evaluated at the end.
MONITOR_VAL_IMAGES = 3000

# Automatically detect the class containing both words.
VMD_CLASS_NAME = None

KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/alzheimer_vgg_checkpoint_bcaf")
FEATURE_DIR = OUTPUT_DIR / "feature_memmaps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# Leave these as None. Set only if automatic discovery cannot find the inputs.
MANUAL_DATA_ROOT = None
MANUAL_CHECKPOINT = None

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Output directory:", OUTPUT_DIR)

# =============================
# 2. Discover Kaggle inputs
# =============================
def is_image_file(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def find_checkpoint():
    if MANUAL_CHECKPOINT is not None:
        path = Path(MANUAL_CHECKPOINT)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    candidates = list(KAGGLE_INPUT.rglob("VGG16_seed42.weights.h5"))
    if not candidates:
        candidates = list(KAGGLE_INPUT.rglob("*.weights.h5"))
    if not candidates:
        candidates = list(KAGGLE_INPUT.rglob("*.h5"))

    if not candidates:
        raise FileNotFoundError(
            "No .h5 checkpoint was found under /kaggle/input. "
            "Add VGG16_seed42.weights.h5 as a Kaggle input."
        )

    print("Checkpoint candidates:")
    for candidate in candidates:
        print(" ", candidate)
    return candidates[0]


def looks_like_class_root(path):
    try:
        subdirs = [p for p in path.iterdir() if p.is_dir()]
        if len(subdirs) < 2:
            return False
        nonempty = 0
        for folder in subdirs:
            if any(is_image_file(p) for p in folder.iterdir()):
                nonempty += 1
        return nonempty >= 2
    except (PermissionError, OSError):
        return False


def find_data_root():
    if MANUAL_DATA_ROOT is not None:
        path = Path(MANUAL_DATA_ROOT)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    roots = []
    for path in [KAGGLE_INPUT] + [p for p in KAGGLE_INPUT.rglob("*") if p.is_dir()]:
        if looks_like_class_root(path):
            roots.append(path)

    if not roots:
        raise FileNotFoundError(
            "Could not find class folders below /kaggle/input. "
            "Set MANUAL_DATA_ROOT near the top of the script."
        )

    def score(root):
        class_dirs = []
        for folder in root.iterdir():
            if folder.is_dir() and any(is_image_file(p) for p in folder.iterdir()):
                class_dirs.append(folder)
        image_count = sum(sum(1 for p in folder.iterdir() if is_image_file(p)) for folder in class_dirs)
        return (len(class_dirs) == NUM_CLASSES_EXPECTED, image_count)

    roots.sort(key=score, reverse=True)
    return roots[0]


CHECKPOINT_PATH = find_checkpoint()
DATA_ROOT = find_data_root()
print("Selected checkpoint:", CHECKPOINT_PATH)
print("Selected dataset root:", DATA_ROOT)

# =============================
# 3. Dataset table and subject IDs
# =============================
def infer_subject_id(path):
    """Extract an OASIS ID; fallback is used only for nonstandard filenames."""
    stem = Path(path).stem
    patterns = [
        r"(OAS\d+[_-]\d+)",
        r"(sub[-_]?OAS\d+[_-]?\d*)",
        r"(OAS\d+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, stem, flags=re.IGNORECASE)
        if match:
            return match.group(1).lower()

    # Fallback. Inspect the printed subject counts if filenames are unusual.
    return re.split(r"[_\- ]", stem)[0].lower()


classes = sorted(
    [
        folder.name
        for folder in DATA_ROOT.iterdir()
        if folder.is_dir() and any(is_image_file(p) for p in folder.iterdir())
    ]
)

if len(classes) != NUM_CLASSES_EXPECTED:
    raise ValueError(
        f"Expected four class folders, found {len(classes)}: {classes}"
    )

class_to_index = {name: i for i, name in enumerate(classes)}
index_to_class = {i: name for name, i in class_to_index.items()}

print("\nClass order used throughout this experiment:")
for i in range(len(classes)):
    print(f"  {i}: {index_to_class[i]}")

rows = []
for class_name in classes:
    files = sorted(
        p for p in (DATA_ROOT / class_name).rglob("*") if is_image_file(p)
    )
    for path in files:
        rows.append(
            {
                "path": str(path),
                "class_name": class_name,
                "label": class_to_index[class_name],
                "subject": infer_subject_id(path),
            }
        )

df_all = pd.DataFrame(rows)
if df_all.empty:
    raise RuntimeError("No images found.")

print("\nAll-image class counts:")
print(df_all["class_name"].value_counts().sort_index())
print("All images:", len(df_all))
print("All subjects:", df_all["subject"].nunique())

# =============================
# 4. Subject-level split first
# =============================
def subject_split(dataframe, seed=SPLIT_SEED):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    train_idx, temporary_idx = next(
        gss1.split(dataframe, groups=dataframe["subject"])
    )
    train = dataframe.iloc[train_idx].copy()
    temporary = dataframe.iloc[temporary_idx].copy()

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    val_rel, test_rel = next(
        gss2.split(temporary, groups=temporary["subject"])
    )
    val = temporary.iloc[val_rel].copy()
    test = temporary.iloc[test_rel].copy()

    train_subjects = set(train["subject"])
    val_subjects = set(val["subject"])
    test_subjects = set(test["subject"])

    assert train_subjects.isdisjoint(val_subjects)
    assert train_subjects.isdisjoint(test_subjects)
    assert val_subjects.isdisjoint(test_subjects)

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


train_raw, val_raw, test_raw = subject_split(df_all)

print("\nRaw subject-level split:")
for name, part in [("train", train_raw), ("validation", val_raw), ("test", test_raw)]:
    print(
        f"{name:12s}: images={len(part):6d}, subjects={part['subject'].nunique():5d}, "
        f"labels={part['label'].value_counts().sort_index().to_dict()}"
    )

# Check subjects per class explicitly, especially Moderate Dementia.
subject_counts = (
    df_all.groupby(["label", "class_name"])["subject"]
    .nunique()
    .reset_index(name="unique_subjects")
)
print("\nUnique subjects per class in complete dataset:")
print(subject_counts.to_string(index=False))

# =============================
# 5. Cap slices within each split
# =============================
def sample_per_subject(dataframe, max_slices=MAX_SLICES_PER_SUBJECT):
    """Keep all subjects and at most max_slices evenly spaced images per subject."""
    def pick(group):
        group = group.sort_values("path").reset_index(drop=True)
        if len(group) <= max_slices:
            return group
        positions = np.linspace(0, len(group) - 1, max_slices)
        positions = np.round(positions).astype(int)
        return group.iloc[positions]

    return (
        dataframe.groupby("subject", group_keys=False)
        .apply(pick)
        .reset_index(drop=True)
    )


train_df = sample_per_subject(train_raw)
val_df = sample_per_subject(val_raw)
test_df = sample_per_subject(test_raw)

# Save exact metadata used by this experiment.
split_metadata = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="validation"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)
split_metadata.to_csv(OUTPUT_DIR / "exact_sampled_subject_split.csv", index=False)

print("\nSampled split used for training/evaluation:")
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(
        f"{name:12s}: images={len(part):6d}, subjects={part['subject'].nunique():5d}, "
        f"labels={part['label'].value_counts().sort_index().to_dict()}"
    )
print("Subject leakage check: PASSED")

# =============================
# 6. Image input pipeline
# =============================
def decode_image(path, label, backbone, training=False):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)

    if training:
        # MRI-compatible conservative augmentation.
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=12.0)
        image = tf.image.random_contrast(image, 0.92, 1.08)
        image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + 16, IMG_SIZE + 16)
        image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])

    if backbone == "VGG16":
        image = tf.keras.applications.vgg16.preprocess_input(image)
    elif backbone == "ResNet50":
        image = tf.keras.applications.resnet50.preprocess_input(image)
    else:
        raise ValueError(backbone)

    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return image, tf.cast(label, tf.int32)


def make_image_dataset(part, backbone, training=False, shuffle=False):
    paths = part["path"].values.astype(str)
    labels = part["label"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(part), 2048), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(
        lambda p, y: decode_image(p, y, backbone, training=training),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(IMAGE_BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

# =============================
# 7. Exact VGG16+CBAM architecture for checkpoint loading
# =============================
@tf.keras.utils.register_keras_serializable(package="AlzheimerCBAM")
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels = int(input_shape[-1])
        hidden = max(channels // self.ratio, 1)
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        self.fc1 = layers.Dense(hidden, activation="relu")
        self.fc2 = layers.Dense(channels)
        self.reshape = layers.Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        a = self.fc2(self.fc1(self.gap(x)))
        b = self.fc2(self.fc1(self.gmp(x)))
        return x * self.reshape(tf.nn.sigmoid(a + b))

    def get_config(self):
        config = super().get_config()
        config.update({"ratio": self.ratio})
        return config


@tf.keras.utils.register_keras_serializable(package="AlzheimerCBAM")
class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            1,
            self.kernel_size,
            padding="same",
            activation="sigmoid",
            use_bias=False,
        )
        super().build(input_shape)

    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx = tf.reduce_max(x, axis=-1, keepdims=True)
        return x * self.conv(tf.concat([avg, mx], axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update({"kernel_size": self.kernel_size})
        return config


# =============================
# 8. Load VGG checkpoint and build ResNet extractor
# =============================
def build_vgg_checkpoint_model():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="VGG16_inp")
    base = VGG16(weights=None, include_top=False, input_tensor=inp)
    base.trainable = False
    x = ChannelAttention(ratio=8, name="VGG16_ca")(base.output)
    x = SpatialAttention(kernel_size=7, name="VGG16_sa")(x)
    x = layers.GlobalAveragePooling2D(name="VGG16_gap")(x)
    x = layers.Dense(256, activation="relu", name="VGG16_fc")(x)
    x = layers.Dropout(0.4, name="VGG16_drop")(x)
    out = layers.Dense(NUM_CLASSES_EXPECTED, activation="softmax", name="VGG16_out")(x)
    return models.Model(inp, out, name="VGG16_CBAM")


vgg_checkpoint_model = build_vgg_checkpoint_model()
print("\nVGG checkpoint model parameters:", f"{vgg_checkpoint_model.count_params():,}")

try:
    vgg_checkpoint_model.load_weights(str(CHECKPOINT_PATH))
    print("VGG16 checkpoint loaded successfully.")
except Exception as error:
    print("Checkpoint loading failed. The uploaded .h5 architecture does not match.")
    raise error

# Reuse the checkpoint's learned attention representation. The checkpoint
# sequence is block5_pool -> VGG16_ca -> VGG16_sa. Extracting VGG16_sa means
# that BCAF receives the learned attention-refined VGG representation.
try:
    vgg_feature_layer = vgg_checkpoint_model.get_layer("VGG16_sa")
except ValueError:
    raise RuntimeError("Could not locate the learned VGG16_sa attention layer.")

vgg_extractor = keras.Model(
    inputs=vgg_checkpoint_model.input,
    outputs=vgg_feature_layer.output,
    name="VGG16_attention_feature_extractor",
)
print("VGG attention feature shape:", vgg_extractor.output_shape)

# ImageNet ResNet50 is the second independent feature branch.
resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
resnet_base.trainable = False
resnet_extractor = keras.Model(
    inputs=resnet_base.input,
    outputs=resnet_base.output,
    name="ResNet50_feature_extractor",
)
print("ResNet feature shape:", resnet_extractor.output_shape)

# =============================
# 9. Baseline VGG checkpoint evaluation
# =============================
def predict_model(model, dataset):
    return model.predict(dataset, verbose=1)


y_val = val_df["label"].values.astype(np.int32)
y_test = test_df["label"].values.astype(np.int32)

vgg_val_ds = make_image_dataset(val_df, "VGG16", training=False, shuffle=False)
vgg_test_ds = make_image_dataset(test_df, "VGG16", training=False, shuffle=False)

vgg_baseline_val_probs = predict_model(vgg_checkpoint_model, vgg_val_ds)
vgg_baseline_test_probs = predict_model(vgg_checkpoint_model, vgg_test_ds)

# =============================
# 10. Stream feature maps to float16 memmaps
# =============================
def extract_to_memmap(extractor, dataset, n_samples, path):
    output_shape = tuple(int(x) for x in extractor.output_shape[1:])
    print(f"Extracting {n_samples} samples -> {path}")
    print("Feature shape:", output_shape)
    print("Approximate float16 size (GB):", n_samples * np.prod(output_shape) * 2 / 1e9)

    mm = np.lib.format.open_memmap(
        path,
        mode="w+",
        dtype=np.float16,
        shape=(n_samples, *output_shape),
    )
    cursor = 0
    for images, _ in dataset:
        features = extractor(images, training=False).numpy().astype(np.float16)
        end = cursor + len(features)
        mm[cursor:end] = features
        cursor = end
        if cursor % (FEATURE_BATCH * 10) == 0 or cursor == n_samples:
            print(f"  {cursor}/{n_samples}")
    if cursor != n_samples:
        raise RuntimeError(f"Expected {n_samples}, extracted {cursor}")
    mm.flush()
    del mm
    gc.collect()


# Use sampled dataframes. Images are not held in memory after extraction.
train_vgg_ds = make_image_dataset(train_df, "VGG16", training=False, shuffle=False)
val_vgg_ds = make_image_dataset(val_df, "VGG16", training=False, shuffle=False)
test_vgg_ds = make_image_dataset(test_df, "VGG16", training=False, shuffle=False)

train_res_ds = make_image_dataset(train_df, "ResNet50", training=False, shuffle=False)
val_res_ds = make_image_dataset(val_df, "ResNet50", training=False, shuffle=False)
test_res_ds = make_image_dataset(test_df, "ResNet50", training=False, shuffle=False)

train_vgg_path = FEATURE_DIR / "train_vgg.npy"
val_vgg_path = FEATURE_DIR / "val_vgg.npy"
test_vgg_path = FEATURE_DIR / "test_vgg.npy"
train_res_path = FEATURE_DIR / "train_resnet.npy"
val_res_path = FEATURE_DIR / "val_resnet.npy"
test_res_path = FEATURE_DIR / "test_resnet.npy"

extract_to_memmap(vgg_extractor, train_vgg_ds, len(train_df), train_vgg_path)
extract_to_memmap(vgg_extractor, val_vgg_ds, len(val_df), val_vgg_path)
extract_to_memmap(vgg_extractor, test_vgg_ds, len(test_df), test_vgg_path)

# Release the VGG model before the larger ResNet extraction.
del train_vgg_ds, val_vgg_ds, test_vgg_ds, vgg_checkpoint_model, vgg_extractor
keras.backend.clear_session()
gc.collect()

extract_to_memmap(resnet_extractor, train_res_ds, len(train_df), train_res_path)
extract_to_memmap(resnet_extractor, val_res_ds, len(val_df), val_res_path)
extract_to_memmap(resnet_extractor, test_res_ds, len(test_df), test_res_path)

del train_res_ds, val_res_ds, test_res_ds, resnet_base, resnet_extractor
keras.backend.clear_session()
gc.collect()

# =============================
# 11. Memmap-backed feature datasets
# =============================
train_vgg_mm = np.load(train_vgg_path, mmap_mode="r")
val_vgg_mm = np.load(val_vgg_path, mmap_mode="r")
test_vgg_mm = np.load(test_vgg_path, mmap_mode="r")
train_res_mm = np.load(train_res_path, mmap_mode="r")
val_res_mm = np.load(val_res_path, mmap_mode="r")
test_res_mm = np.load(test_res_path, mmap_mode="r")


def make_feature_dataset(vgg_mm, res_mm, labels, indices=None, batch=FEATURE_BATCH, shuffle=False, sample_weights=None):
    labels = np.asarray(labels, dtype=np.int32)
    if indices is None:
        indices = np.arange(len(labels), dtype=np.int64)
    else:
        indices = np.asarray(indices, dtype=np.int64)

    if sample_weights is None:
        sample_weights = np.ones(len(labels), dtype=np.float32)
    else:
        sample_weights = np.asarray(sample_weights, dtype=np.float32)

    n = len(indices)

    def generator():
        order = np.random.permutation(n) if shuffle else np.arange(n)
        for start in range(0, n, batch):
            selected = order[start:start + batch]
            source_indices = indices[selected]
            yield (
                (
                    np.asarray(vgg_mm[source_indices], dtype=np.float32),
                    np.asarray(res_mm[source_indices], dtype=np.float32),
                ),
                labels[selected],
                sample_weights[selected],
            )

    signature = (
        (
            tf.TensorSpec(shape=(None, 7, 7, 512), dtype=tf.float32),
            tf.TensorSpec(shape=(None, 7, 7, 2048), dtype=tf.float32),
        ),
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
        tf.TensorSpec(shape=(None,), dtype=tf.float32),
    )
    return tf.data.Dataset.from_generator(generator, output_signature=signature).prefetch(2)


# =============================
# 12. Build the BCAF head
# =============================
@tf.keras.utils.register_keras_serializable(package="AlzheimerBCAF")
class BCAF(layers.Layer):
    def __init__(self, dim=128, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads

    def build(self, input_shapes):
        self.proj_v = layers.Dense(self.dim, use_bias=False, name="proj_v")
        self.proj_r = layers.Dense(self.dim, use_bias=False, name="proj_r")
        self.mha_vr = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.dim // self.num_heads,
            dropout=0.10,
            name="mha_vr",
        )
        self.mha_rv = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.dim // self.num_heads,
            dropout=0.10,
            name="mha_rv",
        )
        self.ln_v = layers.LayerNormalization(name="ln_v")
        self.ln_r = layers.LayerNormalization(name="ln_r")
        super().build(input_shapes)

    def call(self, inputs, training=False):
        f_v, f_r = inputs
        batch_size = tf.shape(f_v)[0]
        height = tf.shape(f_v)[1]
        width = tf.shape(f_v)[2]
        tokens = height * width

        s_v = tf.reshape(self.proj_v(f_v), [batch_size, tokens, self.dim])
        s_r = tf.reshape(self.proj_r(f_r), [batch_size, tokens, self.dim])

        attended_v = self.mha_vr(
            query=s_v, key=s_r, value=s_r, training=training
        )
        attended_r = self.mha_rv(
            query=s_r, key=s_v, value=s_v, training=training
        )

        out_v = self.ln_v(s_v + attended_v)
        out_r = self.ln_r(s_r + attended_r)

        return (
            tf.reshape(out_v, [batch_size, height, width, self.dim]),
            tf.reshape(out_r, [batch_size, height, width, self.dim]),
        )

    def get_config(self):
        config = super().get_config()
        config.update({"dim": self.dim, "num_heads": self.num_heads})
        return config


def build_bcaf_head(n_classes=NUM_CLASSES_EXPECTED, dim=128, heads=4):
    in_v = layers.Input((7, 7, 512), name="vgg_features")
    in_r = layers.Input((7, 7, 2048), name="resnet_features")

    # VGG input is already post-CBAM because it comes from VGG16_sa.
    # Do not apply a second randomly initialized VGG attention block.
    f_v = in_v
    f_r = ChannelAttention(8, name="res_feature_ca")(in_r)
    f_r = SpatialAttention(7, name="res_feature_sa")(f_r)

    fused_v, fused_r = BCAF(dim, heads, name="bcaf")([f_v, f_r])

    x_v = layers.GlobalAveragePooling2D(name="vgg_fused_gap")(fused_v)
    x_r = layers.GlobalAveragePooling2D(name="res_fused_gap")(fused_r)
    x = layers.Concatenate(name="fusion_concat")([x_v, x_r])
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name="fusion_fc",
    )(x)
    x = layers.Dropout(0.40, name="fusion_dropout")(x)
    out = layers.Dense(n_classes, activation="softmax", dtype="float32", name="fusion_out")(x)
    return keras.Model([in_v, in_r], out, name="BCAF_head")


bcaf_model = build_bcaf_head()
print("\nBCAF head parameters:", f"{bcaf_model.count_params():,}")

# =============================
# 13. Monitor validation subset
# =============================
def make_validation_monitor_indices(part, max_images=MONITOR_VAL_IMAGES):
    """Select complete subjects, approximately stratified by class."""
    rng = np.random.default_rng(SEED)
    selected_subjects = []
    total = 0

    # First include subjects from each available class.
    for label in sorted(part["label"].unique()):
        subject_list = part.loc[part["label"] == label, "subject"].unique().tolist()
        rng.shuffle(subject_list)
        if subject_list:
            selected_subjects.append(subject_list[0])

    remaining = [s for s in part["subject"].unique().tolist() if s not in selected_subjects]
    rng.shuffle(remaining)
    selected_subjects.extend(remaining)

    chosen = []
    for subject in selected_subjects:
        count = int((part["subject"] == subject).sum())
        if chosen and total + count > max_images:
            break
        chosen.append(subject)
        total += count
        if total >= max_images:
            break

    indices = np.flatnonzero(part["subject"].isin(chosen).values)
    return indices


val_monitor_indices = make_validation_monitor_indices(val_df)
print("Validation monitor images:", len(val_monitor_indices))
print("Validation full images:", len(val_df))

# =============================
# 14. Class weights and BCAF training
# =============================
y_train = train_df["label"].values.astype(np.int32)
y_val = val_df["label"].values.astype(np.int32)
y_test = test_df["label"].values.astype(np.int32)

present_train_classes = np.unique(y_train)
# Milder clinical weights. The automatic balanced weight for Moderate Dementia
# was 60.5 because only 24 sampled training slices were available. That value
# made the BCAF head over-predict dementia classes and reduced test accuracy.
manual_weight_by_class_name = {
    "Mild Dementia": 2.0,
    "Moderate Dementia": 4.0,
    "Non Demented": 0.5,
    "Very mild Dementia": 1.5,
}
class_weight_map = {
    class_to_index[name]: float(weight)
    for name, weight in manual_weight_by_class_name.items()
    if name in class_to_index
}
train_sample_weights = np.array([class_weight_map[int(y)] for y in y_train], dtype=np.float32)
print("Training class weights:", class_weight_map)

train_feature_ds = make_feature_dataset(
    train_vgg_mm,
    train_res_mm,
    y_train,
    batch=FEATURE_BATCH,
    shuffle=True,
    sample_weights=train_sample_weights,
)
val_monitor_ds = make_feature_dataset(
    val_vgg_mm,
    val_res_mm,
    y_val[val_monitor_indices],
    indices=val_monitor_indices,
    batch=FEATURE_BATCH,
    shuffle=False,
)

bcaf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=BCAF_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

bcaf_checkpoint = OUTPUT_DIR / "BCAF_head_best.weights.h5"
train_callbacks = [
    callbacks.ModelCheckpoint(
        str(bcaf_checkpoint),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    callbacks.CSVLogger(str(OUTPUT_DIR / "bcaf_training_history.csv")),
]

print("\nStarting BCAF head training...")
history = bcaf_model.fit(
    train_feature_ds,
    validation_data=val_monitor_ds,
    epochs=BCAF_EPOCHS,
    callbacks=train_callbacks,
    verbose=1,
)

if bcaf_checkpoint.exists():
    bcaf_model.load_weights(str(bcaf_checkpoint))

bcaf_model.save(OUTPUT_DIR / "BCAF_head_best.keras")
pd.DataFrame(history.history).to_csv(
    OUTPUT_DIR / "bcaf_training_history_full.csv", index=False
)

# =============================
# 15. Full validation/test prediction
# =============================
def predict_feature_dataset(model, vgg_mm, res_mm, labels):
    dataset = make_feature_dataset(
        vgg_mm,
        res_mm,
        labels,
        batch=FEATURE_BATCH,
        shuffle=False,
    )
    return model.predict(dataset, verbose=1)


bcaf_val_probs = predict_feature_dataset(bcaf_model, val_vgg_mm, val_res_mm, y_val)
bcaf_test_probs = predict_feature_dataset(bcaf_model, test_vgg_mm, test_res_mm, y_test)

# =============================
# 16. Metrics
# =============================
def find_vmd_index():
    if VMD_CLASS_NAME is not None:
        return class_to_index.get(VMD_CLASS_NAME)
    matches = [
        name for name in classes
        if "very" in name.lower() and "mild" in name.lower()
    ]
    return class_to_index[matches[0]] if matches else None


VMD_INDEX = find_vmd_index()


def specificity_by_class(y_true, y_pred):
    labels = list(range(NUM_CLASSES_EXPECTED))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    output = {}
    for label, class_name in index_to_class.items():
        tp = cm[label, label]
        fn = cm[label, :].sum() - tp
        fp = cm[:, label].sum() - tp
        tn = cm.sum() - tp - fn - fp
        output[class_name] = float(tn / (tn + fp)) if (tn + fp) else None
    return output


def multiclass_auc(y_true, probs):
    try:
        present = np.unique(y_true)
        if len(present) < 2:
            return None
        return float(
            roc_auc_score(
                y_true,
                probs,
                labels=list(range(NUM_CLASSES_EXPECTED)),
                multi_class="ovr",
                average="macro",
            )
        )
    except Exception:
        return None


def multiclass_average_precision(y_true, probs):
    try:
        present = np.unique(y_true)
        if len(present) < 2:
            return None
        one_hot = tf.keras.utils.to_categorical(y_true, NUM_CLASSES_EXPECTED)
        return float(average_precision_score(one_hot, probs, average="macro"))
    except Exception:
        return None


def evaluate(y_true, probs, split_name, model_name):
    labels = list(range(NUM_CLASSES_EXPECTED))
    preds = np.argmax(probs, axis=1)
    present = np.unique(y_true).tolist()
    present_names = [index_to_class[int(i)] for i in present]

    report_all = classification_report(
        y_true,
        preds,
        labels=labels,
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )
    report_present = classification_report(
        y_true,
        preds,
        labels=present,
        target_names=present_names,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, preds, labels=labels)

    result = {
        "model": model_name,
        "split": split_name,
        "n_images": int(len(y_true)),
        "classes_present": present_names,
        "missing_classes": [index_to_class[i] for i in labels if i not in present],
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy_present_classes": float(
            balanced_accuracy_score(y_true, preds)
        ),
        "macro_f1_present_classes": float(
            f1_score(y_true, preds, labels=present, average="macro", zero_division=0)
        ),
        "macro_f1_all_four_labels": float(
            f1_score(y_true, preds, labels=labels, average="macro", zero_division=0)
        ),
        "log_loss": float(log_loss(y_true, probs, labels=labels)),
        "roc_auc_macro_ovr": multiclass_auc(y_true, probs),
        "average_precision_macro_ovr": multiclass_average_precision(y_true, probs),
        "per_class": {},
        "confusion_matrix_all_four_labels": cm.tolist(),
        "classification_report_present_classes": report_present,
    }

    specificity = specificity_by_class(y_true, preds)
    for label, class_name in index_to_class.items():
        if class_name in report_all:
            result["per_class"][class_name] = {
                "precision": float(report_all[class_name]["precision"]),
                "recall": float(report_all[class_name]["recall"]),
                "f1": float(report_all[class_name]["f1-score"]),
                "specificity": specificity[class_name],
                "status": "evaluated" if label in present else "not_available_in_split",
            }

    if VMD_INDEX is not None and VMD_INDEX in present:
        result["vmd_recall"] = float(report_all[index_to_class[VMD_INDEX]]["recall"])
    else:
        result["vmd_recall"] = None

    path = OUTPUT_DIR / f"{model_name}_{split_name}_metrics.json"
    with open(path, "w") as handle:
        json.dump(result, handle, indent=2)

    print("\n" + "=" * 78)
    print(f"{model_name} — {split_name}")
    print("=" * 78)
    print("Images:", len(y_true))
    print("Classes present:", present_names)
    print("Missing classes:", result["missing_classes"])
    print("Accuracy:", f"{result['accuracy']:.4f}")
    print("Balanced accuracy on present classes:", f"{result['balanced_accuracy_present_classes']:.4f}")
    print("Macro-F1 on present classes:", f"{result['macro_f1_present_classes']:.4f}")
    print("Macro-F1 treating missing classes as zero:", f"{result['macro_f1_all_four_labels']:.4f}")
    print("Macro ROC-AUC:", result["roc_auc_macro_ovr"])
    print("VMD recall:", result["vmd_recall"])
    print("\nClassification report for classes present in this split:")
    print(classification_report(y_true, preds, labels=present, target_names=present_names, zero_division=0))
    print("Confusion matrix, all four labels:")
    print(cm)
    return result


def save_confusion_plot(y_true, probs, title, filename):
    preds = np.argmax(probs, axis=1)
    cm = confusion_matrix(y_true, preds, labels=list(range(NUM_CLASSES_EXPECTED)))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=250)
    plt.close()


# =============================
# 17. Evaluate VGG baseline and BCAF
# =============================
vgg_baseline_val_metrics = evaluate(
    y_val, vgg_baseline_val_probs, "validation", "vgg_checkpoint"
)
vgg_baseline_test_metrics = evaluate(
    y_test, vgg_baseline_test_probs, "test", "vgg_checkpoint"
)
bcaf_val_metrics = evaluate(y_val, bcaf_val_probs, "validation", "bcaf_head")
bcaf_test_metrics = evaluate(y_test, bcaf_test_probs, "test", "bcaf_head")

save_confusion_plot(
    y_test,
    vgg_baseline_test_probs,
    "VGG16+CBAM checkpoint — subject-level test",
    "vgg_checkpoint_test_confusion_matrix.png",
)
save_confusion_plot(
    y_test,
    bcaf_test_probs,
    "BCAF head — subject-level test",
    "bcaf_test_confusion_matrix.png",
)

# Training curves.
history_df = pd.DataFrame(history.history)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df["loss"], label="train")
plt.plot(history_df["val_loss"], label="monitor validation")
plt.title("BCAF training loss")
plt.xlabel("Epoch")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_df["accuracy"], label="train")
plt.plot(history_df["val_accuracy"], label="monitor validation")
plt.title("BCAF training accuracy")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bcaf_training_curves.png", dpi=250)
plt.close()

# =============================
# 18. Save final summary
# =============================
summary = {
    "configuration": {
        "seed": SEED,
        "split_seed": SPLIT_SEED,
        "max_slices_per_subject": MAX_SLICES_PER_SUBJECT,
        "monitor_val_images": MONITOR_VAL_IMAGES,
        "classes": classes,
        "vmd_index": VMD_INDEX,
        "checkpoint": str(CHECKPOINT_PATH),
        "data_root": str(DATA_ROOT),
    },
    "dataset": {
        "all_images": int(len(df_all)),
        "all_subjects": int(df_all["subject"].nunique()),
        "train_images": int(len(train_df)),
        "validation_images": int(len(val_df)),
        "test_images": int(len(test_df)),
        "train_subjects": int(train_df["subject"].nunique()),
        "validation_subjects": int(val_df["subject"].nunique()),
        "test_subjects": int(test_df["subject"].nunique()),
    },
    "vgg_checkpoint_validation": vgg_baseline_val_metrics,
    "vgg_checkpoint_test": vgg_baseline_test_metrics,
    "bcaf_validation": bcaf_val_metrics,
    "bcaf_test": bcaf_test_metrics,
}

with open(OUTPUT_DIR / "final_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

comparison = pd.DataFrame(
    [
        {
            "model": "VGG16 checkpoint",
            "split": "test",
            "accuracy": vgg_baseline_test_metrics["accuracy"],
            "balanced_accuracy_present": vgg_baseline_test_metrics["balanced_accuracy_present_classes"],
            "macro_f1_present": vgg_baseline_test_metrics["macro_f1_present_classes"],
            "macro_roc_auc": vgg_baseline_test_metrics["roc_auc_macro_ovr"],
            "vmd_recall": vgg_baseline_test_metrics["vmd_recall"],
        },
        {
            "model": "BCAF head",
            "split": "test",
            "accuracy": bcaf_test_metrics["accuracy"],
            "balanced_accuracy_present": bcaf_test_metrics["balanced_accuracy_present_classes"],
            "macro_f1_present": bcaf_test_metrics["macro_f1_present_classes"],
            "macro_roc_auc": bcaf_test_metrics["roc_auc_macro_ovr"],
            "vmd_recall": bcaf_test_metrics["vmd_recall"],
        },
    ]
)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
print("\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))

# Save probabilities for future statistical analysis.
np.save(OUTPUT_DIR / "vgg_baseline_val_probs.npy", vgg_baseline_val_probs)
np.save(OUTPUT_DIR / "vgg_baseline_test_probs.npy", vgg_baseline_test_probs)
np.save(OUTPUT_DIR / "bcaf_val_probs.npy", bcaf_val_probs)
np.save(OUTPUT_DIR / "bcaf_test_probs.npy", bcaf_test_probs)

print("\n" + "=" * 78)
print("COMPLETE")
print("=" * 78)
print("All outputs saved to:", OUTPUT_DIR)
print("The test split is final and was not used for training or model selection.")
print("If Moderate Dementia is absent from test, its test metrics are correctly marked unavailable.")
print("Repeat with SPLIT_SEED 123 and 777 before making a publication claim.")


TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Output directory: /kaggle/working/alzheimer_vgg_checkpoint_bcaf
Checkpoint candidates:
  /kaggle/input/datasets/habibleila/vgg16-seed42-weights/VGG16_seed42.weights.h5
Selected checkpoint: /kaggle/input/datasets/habibleila/vgg16-seed42-weights/VGG16_seed42.weights.h5
Selected dataset root: /kaggle/input/datasets/ninadaithal/imagesoasis/Data

Class order used throughout this experiment:
  0: Mild Dementia
  1: Moderate Dementia
  2: Non Demented
  3: Very mild Dementia

All-image class counts:
class_name
Mild Dementia          5002
Moderate Dementia       488
Non Demented          67222
Very mild Dementia    13725
Name: count, dtype: int64
All images: 86437
All subjects: 347

Raw subject-level split:
train       : images= 59414, subjects=  242, labels={0: 3843, 1: 244, 2: 45201, 3: 10126}
validation  : images= 13298, subject